In [1]:
# ============================================================
# MERGED DATASET - DATA PREPROCESSING
# Cleaning + Missing Values + Feature Scaling
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from google.colab import files

# ------------------------------------------------------------
# 1. UPLOAD CSV FILE
# ------------------------------------------------------------

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Original Dataset Shape:", df.shape)
print("\nOriginal Columns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 2. CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = df.columns.str.strip()

# Remove completely empty columns
df = df.dropna(axis=1, how='all')

print("\nColumns after cleaning:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 3. REMOVE DUPLICATE ROWS
# ------------------------------------------------------------

before_duplicates = len(df)

df = df.drop_duplicates()

after_duplicates = len(df)

print("\nDuplicate rows removed:", before_duplicates - after_duplicates)

# ------------------------------------------------------------
# 4. CONVERT DATE COLUMN
# ------------------------------------------------------------

if 'Date' in df.columns:

    df['Date'] = pd.to_datetime(
        df['Date'],
        errors='coerce'
    )

    # Create useful date features
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day

    # Convert date into a numerical representation
    # Number of days from the earliest date
    df['Date_numeric'] = (
        df['Date'] - df['Date'].min()
    ).dt.days

    # Remove original Date column
    df = df.drop(columns=['Date'])

# ------------------------------------------------------------
# 5. CONVERT NUMERIC COLUMNS
# ------------------------------------------------------------

numeric_columns = [
    'Rainfall_mm',
    'Humidity_pct',
    'Temp_C',
    'Windspeed_kmh',
    'Lat',
    'Long'
]

for col in numeric_columns:

    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors='coerce'
        )

# ------------------------------------------------------------
# 6. CHECK MISSING VALUES
# ------------------------------------------------------------

print("\nMissing values BEFORE filling:")
print(df.isnull().sum())

# ------------------------------------------------------------
# 7. HANDLE MISSING VALUES
# ------------------------------------------------------------

# Numeric columns → fill missing values with median
for col in df.select_dtypes(include=np.number).columns:

    df[col] = df[col].fillna(
        df[col].median()
    )

print("\nMissing values AFTER filling:")
print(df.isnull().sum())

# ------------------------------------------------------------
# 8. HANDLE INVALID / IMPOSSIBLE VALUES
# ------------------------------------------------------------

# Rainfall cannot be negative
if 'Rainfall_mm' in df.columns:
    df.loc[df['Rainfall_mm'] < 0, 'Rainfall_mm'] = np.nan

# Humidity should be between 0 and 100
if 'Humidity_pct' in df.columns:
    df.loc[
        (df['Humidity_pct'] < 0) |
        (df['Humidity_pct'] > 100),
        'Humidity_pct'
    ] = np.nan

# Latitude should be between -90 and 90
if 'Lat' in df.columns:
    df.loc[
        (df['Lat'] < -90) |
        (df['Lat'] > 90),
        'Lat'
    ] = np.nan

# Longitude should be between -180 and 180
if 'Long' in df.columns:
    df.loc[
        (df['Long'] < -180) |
        (df['Long'] > 180),
        'Long'
    ] = np.nan

# Windspeed cannot be negative
if 'Windspeed_kmh' in df.columns:
    df.loc[df['Windspeed_kmh'] < 0, 'Windspeed_kmh'] = np.nan

# Temperature - allow negative temperatures,
# because negative temperature can be valid.

# Fill newly created missing values
for col in df.select_dtypes(include=np.number).columns:

    df[col] = df[col].fillna(
        df[col].median()
    )

# ------------------------------------------------------------
# 9. FEATURE SCALING
# ------------------------------------------------------------

# Standardization:
# mean = 0
# standard deviation = 1

scaler = StandardScaler()

columns_to_scale = [
    'Rainfall_mm',
    'Humidity_pct',
    'Temp_C',
    'Windspeed_kmh',
    'Lat',
    'Long'
]

# Scale only columns that actually exist
columns_to_scale = [
    col for col in columns_to_scale
    if col in df.columns
]

df[columns_to_scale] = scaler.fit_transform(
    df[columns_to_scale]
)

# ------------------------------------------------------------
# 10. FINAL DATASET CHECK
# ------------------------------------------------------------

print("\n===================================")
print("PREPROCESSING COMPLETED")
print("===================================")

print("\nFinal Dataset Shape:", df.shape)

print("\nFinal Columns:")
print(df.columns.tolist())

print("\nFinal Missing Values:")
print(df.isnull().sum())

print("\nFirst 5 Rows:")
display(df.head())

# ------------------------------------------------------------
# 11. SAVE PREPROCESSED DATASET
# ------------------------------------------------------------

output_file = "MERGED_DATASET_PREPROCESSED.csv"

df.to_csv(
    output_file,
    index=False
)

print("\nSaved as:", output_file)

# ------------------------------------------------------------
# 12. DOWNLOAD CSV
# ------------------------------------------------------------

files.download(output_file)

Saving MERGED_WEATHER_DATASET.csv to MERGED_WEATHER_DATASET.csv
Original Dataset Shape: (56217, 7)

Original Columns:
['Date', 'Rainfall_mm', 'Humidity_pct', 'Temp_C', 'Windspeed_kmh', 'Lat', 'Long']

Columns after cleaning:
['Date', 'Rainfall_mm', 'Humidity_pct', 'Temp_C', 'Windspeed_kmh', 'Lat', 'Long']

Duplicate rows removed: 0

Missing values BEFORE filling:
Rainfall_mm          0
Humidity_pct         0
Temp_C           52854
Windspeed_kmh    52854
Lat                  0
Long                 0
Year                 0
Month                0
Day                  0
Date_numeric         0
dtype: int64

Missing values AFTER filling:
Rainfall_mm      0
Humidity_pct     0
Temp_C           0
Windspeed_kmh    0
Lat              0
Long             0
Year             0
Month            0
Day              0
Date_numeric     0
dtype: int64

PREPROCESSING COMPLETED

Final Dataset Shape: (56217, 10)

Final Columns:
['Rainfall_mm', 'Humidity_pct', 'Temp_C', 'Windspeed_kmh', 'Lat', 'Long', 'Year', 

,Rainfall_mm,Humidity_pct,Temp_C,Windspeed_kmh,Lat,Long,Year,Month,Day,Date_numeric
0,-0.137679,-0.092869,5.952286,-0.04752,-3.552714e-15,1.421085e-14,2020,5,15,0
1,0.761736,-0.092869,5.952286,-0.04752,-3.552714e-15,1.421085e-14,2020,5,15,0
2,-0.137679,-0.250575,0.005803,-0.04752,-3.552714e-15,1.421085e-14,2020,5,15,0
3,-0.137679,-0.487134,0.005803,-0.04752,-3.552714e-15,1.421085e-14,2020,5,15,0
4,-0.137679,-0.841974,0.005803,-0.04752,-3.552714e-15,1.421085e-14,2020,5,15,0



Saved as: MERGED_DATASET_PREPROCESSED.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# ============================================================
# DISASTER DATASET - PREPROCESSING
# Cleaning + Encoding + Feature Scaling
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder
from google.colab import files


# ============================================================
# 1. UPLOAD DATASET
# ============================================================

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Original Dataset Shape:", df.shape)

print("\nOriginal Columns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# Remove completely empty columns
df = df.dropna(axis=1, how='all')

print("\nColumns after cleaning:")
print(df.columns.tolist())


# ============================================================
# 3. REMOVE DUPLICATE ROWS
# ============================================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("\nDuplicate rows removed:", before - after)


# ============================================================
# 4. CONVERT NUMERICAL COLUMNS
# ============================================================

numeric_columns = [
    'Lat',
    'Long',
    'Magnitude'
]

for col in numeric_columns:

    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors='coerce'
        )


# ============================================================
# 5. CHECK MISSING VALUES
# ============================================================

print("\nMissing values BEFORE preprocessing:")

print(df.isnull().sum())


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

# Numerical columns → median
for col in numeric_columns:

    if col in df.columns:
        df[col] = df[col].fillna(
            df[col].median()
        )


# Categorical columns → mode
categorical_columns = [
    'Disaster_type',
    'Severity'
]

for col in categorical_columns:

    if col in df.columns:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )


# ============================================================
# 7. HANDLE INVALID VALUES
# ============================================================

# Latitude must be between -90 and 90
if 'Lat' in df.columns:

    df.loc[
        (df['Lat'] < -90) |
        (df['Lat'] > 90),
        'Lat'
    ] = np.nan


# Longitude must be between -180 and 180
if 'Long' in df.columns:

    df.loc[
        (df['Long'] < -180) |
        (df['Long'] > 180),
        'Long'
    ] = np.nan


# Magnitude cannot be negative
if 'Magnitude' in df.columns:

    df.loc[
        df['Magnitude'] < 0,
        'Magnitude'
    ] = np.nan


# Fill newly created missing values
for col in numeric_columns:

    if col in df.columns:

        df[col] = df[col].fillna(
            df[col].median()
        )


# ============================================================
# 8. ENCODE DISASTER TYPE
# ============================================================

if 'Disaster_type' in df.columns:

    disaster_encoder = LabelEncoder()

    df['Disaster_type'] = disaster_encoder.fit_transform(
        df['Disaster_type'].astype(str)
    )


# ============================================================
# 9. ENCODE SEVERITY
# ============================================================

if 'Severity' in df.columns:

    # Meaningful order:
    # Light < Moderate < Severe

    severity_mapping = {
        'Light': 0,
        'Moderate': 1,
        'Severe': 2
    }

    df['Severity'] = (
        df['Severity']
        .astype(str)
        .str.strip()
        .map(severity_mapping)
    )

    # If any unexpected severity value exists
    df['Severity'] = df['Severity'].fillna(0)


# ============================================================
# 10. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

scale_columns = [
    'Lat',
    'Long',
    'Magnitude'
]

# Scale only columns that exist
scale_columns = [
    col for col in scale_columns
    if col in df.columns
]

df[scale_columns] = scaler.fit_transform(
    df[scale_columns]
)


# ============================================================
# 11. CHECK FINAL DATASET
# ============================================================

print("\n========================================")
print("PREPROCESSING COMPLETED")
print("========================================")

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nFinal Columns:")
print(df.columns.tolist())

print("\nMissing Values AFTER preprocessing:")

print(df.isnull().sum())

print("\nFinal Dataset:")
display(df.head(10))


# ============================================================
# 12. SAVE PREPROCESSED DATASET
# ============================================================

output_file = "DISASTER_PREPROCESSED.csv"

df.to_csv(
    output_file,
    index=False
)

print("\nFile saved as:", output_file)


# ============================================================
# 13. DOWNLOAD CSV
# ============================================================

files.download(output_file)

Saving DIASTER_DATSET.csv to DIASTER_DATSET.csv
Original Dataset Shape: (261928, 6)

Original Columns:
['Event_id', 'Disaster_type', 'Lat', 'Long', 'Magnitude', 'Severity']

First 5 Rows:


/tmp/ipykernel_3182/3861923030.py:21: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name)


,Event_id,Disaster_type,Lat,Long,Magnitude,Severity
0,EQ-NCS-1,Earthquake,6.572,94.809,5.8,Moderate
1,EQ-NCS-2,Earthquake,6.935,94.613,5.0,Moderate
2,EQ-NCS-3,Earthquake,6.432,94.252,4.7,Light
3,EQ-NCS-4,Earthquake,6.940,94.414,4.8,Light
4,EQ-NCS-5,Earthquake,6.337,94.764,5.0,Moderate



Columns after cleaning:
['Event_id', 'Disaster_type', 'Lat', 'Long', 'Magnitude', 'Severity']

Duplicate rows removed: 0

Missing values BEFORE preprocessing:
Event_id              0
Disaster_type         0
Lat                   0
Long                  0
Magnitude        246849
Severity         246846
dtype: int64

PREPROCESSING COMPLETED

Final Dataset Shape:
(261928, 6)

Final Columns:
['Event_id', 'Disaster_type', 'Lat', 'Long', 'Magnitude', 'Severity']

Missing Values AFTER preprocessing:
Event_id         0
Disaster_type    0
Lat              0
Long             0
Magnitude        0
Severity         0
dtype: int64

Final Dataset:


,Event_id,Disaster_type,Lat,Long,Magnitude,Severity
0,EQ-NCS-1,1,-3.035834,2.277005,-0.056835,1.0
1,EQ-NCS-2,1,-2.963279,2.244273,-0.076522,1.0
2,EQ-NCS-3,1,-3.063817,2.183985,-0.083905,0.0
3,EQ-NCS-4,1,-2.962279,2.211039,-0.081444,0.0
4,EQ-NCS-5,1,-3.082805,2.269490,-0.076522,1.0
5,EQ-NCS-6,1,-3.104192,2.282015,-0.076522,1.0
6,EQ-NCS-7,1,-3.017245,2.253124,-0.064218,1.0
7,EQ-NCS-8,1,0.569744,2.229911,-0.110976,0.0
8,EQ-NCS-9,1,1.610903,-0.085566,-0.120819,0.0
9,EQ-NCS-10,1,-3.004853,2.227740,-0.074061,1.0



File saved as: DISASTER_PREPROCESSED.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# ============================================================
# OSM HOSPITAL / SCHOOL / AMENITY DATASET PREPROCESSING
# Cleaning + Missing Values + Encoding + Feature Scaling
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from google.colab import files


# ============================================================
# 1. UPLOAD DATASET
# ============================================================

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Original Dataset Shape:", df.shape)

print("\nOriginal Columns:")
print(df.columns.tolist())

display(df.head())


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# Remove completely empty columns
df = df.dropna(axis=1, how='all')


# ============================================================
# 3. REMOVE DUPLICATE ROWS
# ============================================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("\nDuplicate rows removed:", before - after)


# ============================================================
# 4. CONVERT NUMERICAL COLUMNS
# ============================================================

numeric_columns = [
    'ID',
    'Lat',
    'Long',
    'Hospital_Count'
]

for col in numeric_columns:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors='coerce'
        )


# ============================================================
# 5. CHECK MISSING VALUES
# ============================================================

print("\nMissing values BEFORE handling:")

print(df.isnull().sum())


# ============================================================
# 6. HANDLE NUMERICAL MISSING VALUES
# ============================================================

for col in numeric_columns:

    if col in df.columns:

        df[col] = df[col].fillna(
            df[col].median()
        )


# ============================================================
# 7. HANDLE CATEGORICAL MISSING VALUES
# ============================================================

categorical_columns = [
    'Type',
    'School_Name',
    'Amenity',
    'OSM_Type'
]

for col in categorical_columns:

    if col in df.columns:

        df[col] = df[col].fillna('Unknown')


# ============================================================
# 8. CLEAN TEXT VALUES
# ============================================================

for col in categorical_columns:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )


# ============================================================
# 9. ENCODE CATEGORICAL COLUMNS
# ============================================================

encoders = {}

for col in categorical_columns:

    if col in df.columns:

        encoder = LabelEncoder()

        df[col] = encoder.fit_transform(
            df[col]
        )

        encoders[col] = encoder

        print("\n", col, "Mapping:")

        for number, category in enumerate(
            encoder.classes_
        ):

            print(category, "→", number)


# ============================================================
# 10. CHECK LATITUDE / LONGITUDE
# ============================================================

if 'Lat' in df.columns:

    df.loc[
        (df['Lat'] < -90) |
        (df['Lat'] > 90),
        'Lat'
    ] = np.nan


if 'Long' in df.columns:

    df.loc[
        (df['Long'] < -180) |
        (df['Long'] > 180),
        'Long'
    ] = np.nan


# Fill invalid coordinates
for col in ['Lat', 'Long']:

    if col in df.columns:

        df[col] = df[col].fillna(
            df[col].median()
        )


# ============================================================
# 11. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

scale_columns = [
    'Lat',
    'Long',
    'Hospital_Count'
]

scale_columns = [
    col for col in scale_columns
    if col in df.columns
]

df[scale_columns] = scaler.fit_transform(
    df[scale_columns]
)


# ============================================================
# 12. DO NOT SCALE ID
# ============================================================

# ID is only an identifier.
# We keep it unchanged.


# ============================================================
# 13. FINAL DATASET CHECK
# ============================================================

print("\n==========================================")
print("PREPROCESSING COMPLETED")
print("==========================================")

print("\nFinal Shape:")
print(df.shape)

print("\nFinal Columns:")
print(df.columns.tolist())

print("\nMissing Values AFTER preprocessing:")

print(df.isnull().sum())

print("\nFinal Dataset:")
display(df.head(10))


# ============================================================
# 14. SAVE PREPROCESSED DATASET
# ============================================================

output_file = "OSM_PREPROCESSED.csv"

df.to_csv(
    output_file,
    index=False
)

print("\nSaved as:", output_file)


# ============================================================
# 15. DOWNLOAD CSV
# ============================================================

files.download(output_file)

Saving HOSPITAL_SCHOOL_MERGED.csv to HOSPITAL_SCHOOL_MERGED.csv
Original Dataset Shape: (55669, 8)

Original Columns:
['ID', 'Lat', 'Long', 'Type', 'Hospital_Count', 'School_Name', 'Amenity', 'OSM_Type']


,ID,Lat,Long,Type,Hospital_Count,School_Name,Amenity,OSM_Type
0,198490585,30.888420,75.791530,Hospital,1,NaN,NaN,NaN
1,243415888,30.840700,75.979575,Hospital,1,NaN,NaN,NaN
2,245638783,16.886275,78.960869,Hospital,1,NaN,NaN,NaN
3,247177822,11.100545,76.963745,Hospital,1,NaN,NaN,NaN
4,248420920,13.038649,80.205216,Hospital,1,NaN,NaN,NaN



Duplicate rows removed: 0

Missing values BEFORE handling:
ID                    0
Lat                   0
Long                  0
Type                  0
Hospital_Count        0
School_Name       55631
Amenity           55629
OSM_Type          55629
dtype: int64

 Type Mapping:
hospital → 0
school → 1

 School_Name Mapping:
amrit special needs school → 0
amrita vidhyalayam (matriculation) → 1
angappa hr. sec. school (cbse) → 2
angappa matric school → 3
avinashilingam vocational training institute → 4
bishop ambrose college → 5
bishop ubagarm's primary school → 6
brindisi matric. ir. sec. school → 7
cms matriculation higher secondary school → 8
college of pharamacy → 9
corporation boys high school → 10
corporation elementary school → 11
corporation girls high school → 12
corporaton school for deaf → 13
ece department → 14
kattur municipal elementary school → 15
kikani higher secondary school → 16
krishnaswamy nagar - ganga nagar matriculation school → 17
mani higher secondary school →

,ID,Lat,Long,Type,Hospital_Count,School_Name,Amenity,OSM_Type
0,198490585,1.691998,-0.462840,0,0.156749,34,1,1
1,243415888,1.684708,-0.416098,0,0.156749,34,1,1
2,245638783,-0.447120,0.324951,0,0.156749,34,1,1
3,247177822,-1.331010,-0.171467,0,0.156749,34,1,1
4,248420920,-1.034925,0.634254,0,0.156749,34,1,1
5,248560797,-1.022073,0.656834,0,0.156749,34,1,1
6,249132373,1.319627,-0.140692,0,0.156749,34,1,1
7,258978416,-1.026744,0.652666,0,0.156749,34,1,1
8,258978417,-1.025781,0.652747,0,0.156749,34,1,1
9,262452334,-1.027938,0.636956,0,0.156749,34,1,1



Saved as: OSM_PREPROCESSED.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# ============================================================
# POPULATION DATASET - PREPROCESSING
# Cleaning + Missing Values + Feature Scaling
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from google.colab import files


# ============================================================
# 1. UPLOAD DATASET
# ============================================================

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Original Dataset Shape:", df.shape)

print("\nOriginal Columns:")
print(df.columns.tolist())

print("\nOriginal Dataset:")
display(df.head())


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# Remove completely empty columns
df = df.dropna(axis=1, how='all')

print("\nColumns after cleaning:")
print(df.columns.tolist())


# ============================================================
# 3. REMOVE DUPLICATE ROWS
# ============================================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("\nDuplicate rows removed:", before - after)


# ============================================================
# 4. CONVERT NUMERICAL COLUMNS
# ============================================================

numeric_columns = [
    'Pop',
    'Pop_den',
    'Lat',
    'Long'
]

for col in numeric_columns:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors='coerce'
        )


# ============================================================
# 5. CHECK MISSING VALUES
# ============================================================

print("\nMissing values BEFORE handling:")

print(df.isnull().sum())


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

# Replace missing numerical values
# with the median of that column

for col in numeric_columns:

    if col in df.columns:

        df[col] = df[col].fillna(
            df[col].median()
        )


# ============================================================
# 7. CHECK INVALID LATITUDE
# ============================================================

if 'Lat' in df.columns:

    df.loc[
        (df['Lat'] < -90) |
        (df['Lat'] > 90),
        'Lat'
    ] = np.nan


# ============================================================
# 8. CHECK INVALID LONGITUDE
# ============================================================

if 'Long' in df.columns:

    df.loc[
        (df['Long'] < -180) |
        (df['Long'] > 180),
        'Long'
    ] = np.nan


# Fill any missing values created above
for col in ['Lat', 'Long']:

    if col in df.columns:

        df[col] = df[col].fillna(
            df[col].median()
        )


# ============================================================
# 9. CHECK POPULATION VALUES
# ============================================================

# Population cannot be negative

if 'Pop' in df.columns:

    df.loc[
        df['Pop'] < 0,
        'Pop'
    ] = np.nan


# Population density cannot be negative

if 'Pop_den' in df.columns:

    df.loc[
        df['Pop_den'] < 0,
        'Pop_den'
    ] = np.nan


# Fill invalid values
for col in ['Pop', 'Pop_den']:

    if col in df.columns:

        df[col] = df[col].fillna(
            df[col].median()
        )


# ============================================================
# 10. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

scale_columns = [
    'Pop',
    'Pop_den',
    'Lat',
    'Long'
]

df[scale_columns] = scaler.fit_transform(
    df[scale_columns]
)


# ============================================================
# 11. FINAL CHECK
# ============================================================

print("\n==========================================")
print("PREPROCESSING COMPLETED")
print("==========================================")

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nFinal Columns:")
print(df.columns.tolist())

print("\nMissing Values AFTER preprocessing:")

print(df.isnull().sum())

print("\nPreprocessed Dataset:")
display(df.head(10))


# ============================================================
# 12. SAVE PREPROCESSED DATASET
# ============================================================

output_file = "POPULATION_PREPROCESSED.csv"

df.to_csv(
    output_file,
    index=False
)

print("\nSaved as:", output_file)


# ============================================================
# 13. DOWNLOAD CSV
# ============================================================

files.download(output_file)

Saving Population_2000_2020_Aggregated.csv to Population_2000_2020_Aggregated.csv
Original Dataset Shape: (341251, 4)

Original Columns:
['Pop', 'Pop_den', 'Lat', 'Long']

Original Dataset:


,Pop,Pop_den,Lat,Long
0,152.60,25.5661,6.75,93.80
1,197.05,33.0157,6.75,93.85
2,395.74,30.9453,6.80,93.80
3,401.77,19.6362,6.80,93.85
4,463.42,41.8127,6.80,93.90



Columns after cleaning:
['Pop', 'Pop_den', 'Lat', 'Long']

Duplicate rows removed: 17

Missing values BEFORE handling:
Pop        0
Pop_den    0
Lat        0
Long       0
dtype: int64

PREPROCESSING COMPLETED

Final Dataset Shape:
(341234, 4)

Final Columns:
['Pop', 'Pop_den', 'Lat', 'Long']

Missing Values AFTER preprocessing:
Pop        0
Pop_den    0
Lat        0
Long       0
dtype: int64

Preprocessed Dataset:


,Pop,Pop_den,Lat,Long
0,-0.370211,-0.346004,-2.884294,2.360175
1,-0.368564,-0.338443,-2.884294,2.368536
2,-0.361200,-0.340544,-2.875344,2.360175
3,-0.360977,-0.352022,-2.875344,2.368536
4,-0.358692,-0.329515,-2.875344,2.376897
5,-0.365839,-0.362190,-2.866394,2.360175
6,-0.365260,-0.362488,-2.866394,2.368536
7,-0.354274,-0.341794,-2.866394,2.376897
8,-0.368683,-0.350972,-2.857445,2.351814
9,-0.372142,-0.368628,-2.857445,2.360175



Saved as: POPULATION_PREPROCESSED.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>